# On nettoie les données DERA

Ici, on prépare les données privées et publiques avec les règles de `Nettoyage data`.

Les fichiers de départ ne bougent pas : les versions nettoyées iront dans `Data nettoyees/`. On regarde aussi les valeurs manquantes et les doublons, sans supprimer de lignes automatiquement.

## 1. On prépare le projet

On repère le dossier du projet et les deux fichiers à traiter. Tu peux lancer le notebook depuis la racine du projet ou un de ses sous-dossiers.

In [ ]:
from pathlib import Path

import pandas as pd

def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data brutes').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data brutes » introuvable. Lancez le notebook depuis le projet.'
    )


RACINE = trouver_racine_projet()
DOSSIER_BRUT = RACINE / 'Data brutes'
DOSSIER_SORTIE = RACINE / 'Data nettoyees'
FICHIERS = {
    'prive': DOSSIER_BRUT / 'DS_DERA_PRIVE_SERIES_LONGUES_data.csv',
    'public': DOSSIER_BRUT / 'DS_DERA_PUBLIC_SERIES_LONGUES_data.csv',
}

for nom, chemin in FICHIERS.items():
    if not chemin.is_file():
        raise FileNotFoundError(f'Fichier brut manquant ({nom}) : {chemin}')

print(f'Projet : {RACINE}')
print(f'Pandas : {pd.__version__}')

Projet : /Users/yanis.yvray/ProjetPython
Pandas : 2.0.3


## 2. On prépare les noms et les catégories

On reprend les noms et traductions de `Nettoyage data`. Si un code n’est pas dans la liste, on le garde tel quel.

In [2]:
RENOMMAGE = {
    'GEO': 'geo',
    'GEO_OBJECT': 'zone_geo',
    'FREQ': 'frequence',
    'ACTIVITY': 'secteur_activite',
    'AGE': 'tranche_age',
    'NUMBER_EMPL': 'taille_entreprise',
    'PCS_ESE': 'categorie_socio_prof',
    'SEX': 'sexe',
    'WKTIME': 'temps_travail',
    'QUANTILE': 'centile_salaire',
    'DERA_MEASURE': 'indicateur',
    'UNIT_MEASURE': 'unite',
    'TIME_PERIOD': 'annee',
    'OBS_VALUE': 'valeur',
    'CIVILSERVANT_STATUS': 'statut_fonctionnaire',
    'PUBLIC_LEGAL_FORM': 'forme_juridique_public',
}

MODALITES = {
    'sexe': {'F': 'Femme', 'M': 'Homme', '_T': 'Ensemble'},
    'categorie_socio_prof': {
        '3': 'Cadres et prof. intellectuelles sup.',
        '4': 'Professions intermédiaires',
        '5': 'Employés',
        '6': 'Ouvriers',
        '_T': 'Ensemble',
    },
    'tranche_age': {
        'Y_LT26': 'Moins de 26 ans',
        'Y26T30': 'De 26 à 30 ans',
        'Y31T40': 'De 31 à 40 ans',
        'Y41T50': 'De 41 à 50 ans',
        'Y51T60': 'De 51 à 60 ans',
        'Y_GT60': 'Plus de 60 ans',
        '_T': 'Ensemble',
    },
    'taille_entreprise': {
        'E_LT10': 'Moins de 10 salariés',
        'E10T49': '10 à 49 salariés',
        'E50T199': '50 à 199 salariés',
        'E200T499': '200 à 499 salariés',
        'E500T999': '500 à 999 salariés',
        'E_GE1000': '1000 salariés et plus',
        '_T': 'Ensemble',
    },
    'unite': {
        'XDC': 'Euros constants',
        'PT': 'Pourcentage',
        'IX': 'Indice base 100',
    },
    'temps_travail': {'FT': 'Temps complet', '_T': 'Ensemble'},
    'secteur_activite': {
        'H': 'Transports et entreposage',
        'I': 'Hébergement et restauration',
        'J': 'Information et communication',
        'K': 'Activités financières et d’assurance',
        'L': 'Activités immobilières',
        'M_N': 'Activités spécialisées, scientifiques, techniques et services support (M_N)',
        'OTQ': 'Administration publique, enseignement, santé et action sociale (OTQ)',
        'R_S': 'Arts, divertissement, loisirs et autres services (R_S)',
        'B_D_E': 'Industries extractives, énergie, eau, déchets (B_D_E)',
        'C10T12': 'Fabrication d’aliments, boissons et tabac (C10T12)',
        'C19': 'Cokéfaction et raffinage (C19)',
        'C26T28': 'Fabrication de produits informatiques, électroniques et machines (C26T28)',
        'C29_30': 'Fabrication de matériels de transport (C29_30)',
        'C_OTH': 'Autres industries manufacturières (C_OTH)',
        'F': 'Construction',
        'G': 'Commerce ; réparation d’automobiles et de motocycles',
        '_T': 'Ensemble des secteurs',
    },
    'indicateur': {
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_EUROS_CONSTANTS': 'Salaire net moyen mensuel (Euros constants)',
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_EVOLUTION_CONSTANT': 'Évolution du salaire net mensuel (%)',
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_INDICE_100_1996': 'Indice du salaire net mensuel (Base 100 en 1996)',
        'PARTMS_10': 'Part de la masse salariale perçue par le top 10%',
        'PARTMS_100': 'Part de la masse salariale perçue par le top 1%',
        'PARTMS_1000': 'Part de la masse salariale perçue par le top 0,1%',
        'PARTMS_10000': 'Part de la masse salariale perçue par le top 0,01%',
    },
    'centile_salaire': {
        'CENTILE_10': '10e centile (D1 - 10% gagnent moins)',
        'CENTILE_25': '1er quartile (Q1 - 25% gagnent moins)',
        'CENTILE_50': 'Médiane (50% gagnent moins/plus)',
        'CENTILE_75': '3e quartile (Q3 - 75% gagnent moins)',
        'CENTILE_90': '90e centile (D9 - 10% gagnent plus)',
        'CENTILE_95': '95e centile (Top 5%)',
        'CENTILE_99': '99e centile (Top 1%)',
        '_T': 'Ensemble',
    },
}

## 3. La fonction de nettoyage

La fonction enlève les espaces en trop, renomme les colonnes et vérifie que l’année et la valeur sont bien numériques. Elle prépare aussi un petit bilan. Pas de suppression de lignes ici.

In [3]:
def nettoyer_donnees(df, nom_jeu):
    donnees = df.copy()
    donnees.columns = donnees.columns.str.strip().str.upper()

    colonnes_requises = {'TIME_PERIOD', 'OBS_VALUE'}
    manquantes = colonnes_requises.difference(donnees.columns)
    if manquantes:
        raise ValueError(
            f'{nom_jeu} : colonnes obligatoires absentes : {sorted(manquantes)}'
        )

    for colonne in donnees.select_dtypes(include=['object']).columns:
        donnees[colonne] = donnees[colonne].str.strip().replace('', pd.NA)

    donnees = donnees.rename(columns=RENOMMAGE)
    invalides = {}
    for colonne in ('annee', 'valeur'):
        numerique = pd.to_numeric(donnees[colonne], errors='coerce')
        invalides[colonne] = int((donnees[colonne].notna() & numerique.isna()).sum())
        donnees[colonne] = numerique

    donnees['annee'] = donnees['annee'].astype('Int64')

    for colonne, recodage in MODALITES.items():
        if colonne in donnees.columns:
            donnees[colonne] = donnees[colonne].replace(recodage)

    bilan = {
        'jeu': nom_jeu,
        'lignes': len(donnees),
        'colonnes': len(donnees.columns),
        'valeurs_manquantes': int(donnees.isna().sum().sum()),
        'doublons_exacts': int(donnees.duplicated().sum()),
        'annees_invalides': invalides['annee'],
        'valeurs_invalides': invalides['valeur'],
    }
    return donnees, bilan

## 4. On lance le nettoyage

Les fichiers utilisent `;` entre les colonnes, donc on garde le même format pour les résultats. Les CSV nettoyés s’ouvrent aussi plus facilement dans un tableur.

In [4]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

donnees_nettoyees = {}
bilans = []

for nom_jeu, chemin in FICHIERS.items():
    donnees_brutes = pd.read_csv(chemin, sep=';', encoding='utf-8-sig')
    donnees, bilan = nettoyer_donnees(donnees_brutes, nom_jeu)
    donnees_nettoyees[nom_jeu] = donnees
    bilans.append(bilan)

    chemin_sortie = DOSSIER_SORTIE / f'{chemin.stem}_nettoye.csv'
    donnees.to_csv(chemin_sortie, sep=';', index=False, encoding='utf-8-sig')
    print(f'{nom_jeu} : export créé -> {chemin_sortie}')

bilan_qualite = pd.DataFrame(bilans).set_index('jeu')
display(bilan_qualite)

prive : export créé -> /Users/yanis.yvray/ProjetPython/Data nettoyees/DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv
public : export créé -> /Users/yanis.yvray/ProjetPython/Data nettoyees/DS_DERA_PUBLIC_SERIES_LONGUES_data_nettoye.csv


,lignes,colonnes,valeurs_manquantes,doublons_exacts,annees_invalides,valeurs_invalides
jeu,,,,,,
prive,14768,14,0,0,0,0
public,2183,13,0,0,0,0


## 5. Petit coup d’œil au résultat

On affiche quelques lignes pour voir si tout s’est bien passé. Les codes qu’on ne connaît pas restent comme dans le fichier de départ.

In [5]:
for nom_jeu, donnees in donnees_nettoyees.items():
    print(f'\nAperçu — {nom_jeu}')
    display(donnees.head())

print(f'\nFichiers nettoyés enregistrés dans : {DOSSIER_SORTIE}')


Aperçu — prive


,geo,zone_geo,frequence,secteur_activite,tranche_age,taille_entreprise,categorie_socio_prof,sexe,temps_travail,centile_salaire,indicateur,unite,annee,valeur
0,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Femme,Temps complet,Ensemble,Salaire net moyen mensuel (Euros constants),Euros constants,1961,816.00
1,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,Ensemble,Salaire net moyen mensuel (Euros constants),Euros constants,1952,826.00
2,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,Ensemble,Évolution du salaire net mensuel (%),Pourcentage,1973,4.75
3,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,99e centile (Top 1%),Salaire net moyen mensuel (Euros constants),Euros constants,2010,11286.00
4,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Femme,Temps complet,Ensemble,Évolution du salaire net mensuel (%),Pourcentage,1963,5.60



Aperçu — public


,tranche_age,statut_fonctionnaire,indicateur,frequence,geo,zone_geo,forme_juridique_public,centile_salaire,sexe,unite,temps_travail,annee,valeur
0,Ensemble,_T,Salaire net moyen mensuel (Euros constants),A,F,FRANCE,_T,CENTILE_60,Ensemble,Euros constants,Ensemble,2015,2548.00
1,Ensemble,_T,Salaire net moyen mensuel (Euros constants),A,F,FRANCE,_T,CENTILE_70,Ensemble,Euros constants,Ensemble,2022,2862.00
2,Ensemble,_T,Évolution du salaire net mensuel (%),A,F,FRANCE,_T,CENTILE_40,Ensemble,Pourcentage,Ensemble,2022,-1.34
3,Ensemble,F_B,Salaire net moyen mensuel (Euros constants),A,F,FRANCE,3,Ensemble,Ensemble,Euros constants,Ensemble,2021,2881.00
4,Ensemble,_T,Salaire net moyen mensuel (Euros constants),A,F,FRANCE,2,CENTILE_60,Ensemble,Euros constants,Ensemble,2015,2180.00



Fichiers nettoyés enregistrés dans : /Users/yanis.yvray/ProjetPython/Data nettoyees
